# Three-Way AI Conversation

**Author:** Steve A.

## Description

This notebook extends the Week 2 Day 1 exercise by creating a conversation between three AI models:

- OpenAI
- Google Gemini
- OpenRouter

Each model has its own system prompt and responds to the ongoing conversation.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from google import genai
from IPython.display import display, Markdown
# Load API keys and other configuration from the project's .env file
load_dotenv()

# Create a client that communicates directly with the OpenAI API
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Create a client using Google's Gemini SDK
gemini_client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

# OpenRouter provides an OpenAI-compatible API, so we reuse the OpenAI client
# while changing the server address and API key
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [ ]:
# Models used by each provider
OPENAI_MODEL = "gpt-5-mini"
GEMINI_MODEL =  "gemini-3.5-flash"
OPENROUTER_MODEL = "openrouter/free"

In [ ]:
# Create a client that communicates directly with OpenAI
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Create a client using Google's Gemini SDK
gemini_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# OpenRouter uses an OpenAI-compatible API
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [ ]:
openai_system = """
You are OpenAI.
You are witty, playful, and naturally funny.
You make clever jokes while still contributing useful ideas to the discussion.
Do not let humor replace substance.
"""

gemini_system = """
You are Gemini.
You are blunt, sarcastic, and slightly rude.
You challenge weak ideas directly and do not sugarcoat your opinions.
Stay entertaining, but do not become hateful or personally abusive.
"""

openrouter_system = """
You are OpenRouter.
You are cheerful, enthusiastic, and relentlessly optimistic.
You look for the positive side of every idea and encourage the other participants.
Your energy should feel bright and upbeat without sounding childish.
"""

In [ ]:
conversation_topic = """
Discuss whether artificial intelligence will make people more creative
or make them overly dependent on technology.
"""

In [ ]:
openai_messages = [conversation_topic]
gemini_messages = []
openrouter_messages = []

In [ ]:
# Build the conversation that will be sent to OpenAI
def call_openai():
    # Build the conversation context that OpenAI will receive
    messages = [
        {"role": "system", "content": openai_system},
        {"role": "user", "content": conversation_topic}
    ]

    # Send the conversation context to the selected OpenAI model
    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=messages
    )

    # Extract and return the model's written response
    return response.choices[0].message.content

In [ ]:
def call_gemini():
    # Build the conversation that will be sent to Gemini
    messages = [
        {"role": "system", "content": gemini_system},
        {
            "role": "user",
            "content": f"""
The discussion topic is:

{conversation_topic}

OpenAI said:

{openai_messages[-1]}

Respond directly to OpenAI and continue the discussion.
"""
        }
    ]

    # Send the conversation to Gemini
    response = gemini_client.chat.completions.create(
        model=GEMINI_MODEL,
        messages=messages
    )

    # Return only Gemini's written response
    return response.choices[0].message.content

In [ ]:
def call_openrouter():
    # Build the conversation that will be sent to OpenRouter
    messages = [
        {"role": "system", "content": openrouter_system}
    ]

    for openai, openrouter in zip(openai_messages, openrouter_messages):
        messages.append({"role": "assistant", "content": openai})
        messages.append({"role": "user", "content": openrouter})

    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
# Run the three-way conversation
for turn in range(5):
    display(Markdown(f"# Turn {turn + 1}"))

    # OpenAI responds
    openai_reply = call_openai()
    openai_messages.append(openai_reply)

    display(
        Markdown(
            f"## 🤖 OpenAI\n\n"
            f"{openai_reply}\n\n"
            f"---"
        )
    )

    # Gemini responds
    gemini_reply = call_gemini()
    gemini_messages.append(gemini_reply)

    display(
        Markdown(
            f"## 💎 Gemini\n\n"
            f"{gemini_reply}\n\n"
            f"---"
        )
    )

    # OpenRouter responds
    openrouter_reply = call_openrouter()
    openrouter_messages.append(openrouter_reply)

    display(
        Markdown(
            f"## 🌐 OpenRouter\n\n"
            f"{openrouter_reply}\n\n"
            f"---"
        )
    )